In [ ]:
# Load the clean table and check it against the SQL results from Day 135
import os
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
P = "/content/drive/MyDrive/Project3_Procurement/"
df = pd.read_parquet(P + "awards_clean.parquet")
D = P + "dash/"
os.makedirs(D, exist_ok=True)
tot = df["amount_usd"].sum()
assert len(df) == 4839
assert abs(tot - 2145458626) < 2
firms = df[df["is_individual"] == 0]
s = (firms.groupby("supplier_key")["amount_usd"]
     .sum().sort_values(ascending=False))
s = s[s > 0]
cum = s.cumsum() / s.sum()
n80 = int((cum < 0.8).sum() + 1)
top10 = s.head(10).sum() / s.sum()
assert len(s) == 2931
assert n80 == 180
assert round(top10, 3) == 0.426
w = lambda m: df.loc[m, "amount_usd"].sum()
full = w(df["tier"] == "Full") / tot
direct = w(df["tier"] == "Direct") / tot
print(len(s), n80, round(top10, 3),
      round(full, 3), round(direct, 3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2931 180 0.426 0.798 0.136


In [ ]:
# UN agencies (country code 1W) and the Goods category with and without them
un = df["country"].eq("1W")
print(df[un].groupby("supplier_key")["amount_usd"]
      .sum().sort_values(ascending=False)
      .head(5).round())
ex = df[~un]
tot_ex = ex["amount_usd"].sum()
dir_ex = ex.loc[ex["tier"] == "Direct",
                "amount_usd"].sum()
print(round(dir_ex), round(dir_ex / tot_ex, 3))
gf = lambda d: (d.loc[d["tier"] == "Full",
    "amount_usd"].sum() / d["amount_usd"].sum())
g = df[df["category"] == "Goods"]
ge = g[g["country"].ne("1W")]
print(round(gf(g), 3), round(gf(ge), 3))

supplier_key
UNITED NATIONS CHILDREN S FUND UNICEF | 1W                    144158703.0
UNITED NATIONS HUMAN SETTLEMENTS PROGRAMME UN HABITAT | 1W      5959620.0
UNITED NATIONS OFFICE PROJECT SERVICES UNOPS | 1W                 39637.0
Name: amount_usd, dtype: float64
141142504 0.071
0.446 0.634


In [ ]:
# Four levers (value at stake per 1%); they overlap, so never add them up
l1 = ex[ex["tier"] == "Direct"]
lim = df[(df["tier"] == "Limited")
         & (df["is_individual"] == 0)]
gl = lim.groupby("supplier_key")["amount_usd"].agg(
    awards="size", spend="sum")
l2 = gl[gl["awards"] >= 5]
tail = s.iloc[n80:]
l4 = df.nlargest(5, "amount_usd")
opps = pd.DataFrame([
 ("Direct awards outside UN agencies",
  len(l1), "awards", l1["amount_usd"].sum()),
 ("Suppliers with 5+ small awards",
  len(l2), "suppliers", l2["spend"].sum()),
 ("Tail: suppliers beyond the 80% line",
  len(tail), "suppliers", tail.sum()),
 ("Five largest awards",
  5, "awards", l4["amount_usd"].sum())],
 columns=["lever", "count", "unit", "spend_usd"])
opps["spend_m"] = (opps["spend_usd"] / 1e6).round(2)
opps["per_1pct_m"] = (opps["spend_usd"] / 1e8).round(2)
opps["share_of_total"] = opps["spend_usd"] / tot
print(opps.round(3))
assert abs(opps.loc[3, "spend_usd"] / tot
           - 0.282) < 0.001

                                 lever  count       unit     spend_usd  \
0    Direct awards outside UN agencies    381     awards  1.411425e+08   
1       Suppliers with 5+ small awards     27  suppliers  8.324447e+06   
2  Tail: suppliers beyond the 80% line   2751  suppliers  4.231466e+08   
3                  Five largest awards      5     awards  6.059619e+08   

   spend_m  per_1pct_m  share_of_total  
0   141.14        1.41           0.066  
1     8.32        0.08           0.004  
2   423.15        4.23           0.197  
3   605.96        6.06           0.282  


In [ ]:
# Five small tables for Looker Studio, saved in the dash folder
def label(k):
    n, c = k.rsplit(" | ", 1)
    if "UNICEF" in n:
        return "UNICEF (UN agency)"
    return n.title()[:42]
kpi = pd.DataFrame([{
 "total_m": round(tot / 1e6, 1),
 "suppliers": len(s), "n_for_80": n80,
 "pct_suppliers_for_80": n80 / len(s),
 "top10_share": top10, "full_share": full,
 "direct_share": direct,
 "direct_share_ex_un": dir_ex / tot_ex}])
par = pd.DataFrame({"rank": np.arange(1, 301),
    "cum_pct": cum.values[:300]})
ct = (df.groupby(["category", "tier"])
      ["amount_usd"].sum().reset_index())
ct["spend_m"] = (ct["amount_usd"] / 1e6).round(2)
ct["share_of_total"] = ct["amount_usd"] / tot
t10 = s.head(10).reset_index()
t10.columns = ["key", "spend"]
t10["supplier"] = t10["key"].map(label)
t10["spend_m"] = (t10["spend"] / 1e6).round(2)
t10["share_of_firm_spend"] = t10["spend"] / s.sum()
assert abs(ct["amount_usd"].sum() - tot) < 1
kpi.to_csv(D + "kpi.csv", index=False)
par.to_csv(D + "pareto.csv", index=False)
ct[["category", "tier", "spend_m",
    "share_of_total"]].to_csv(
    D + "category_tier.csv", index=False)
t10[["supplier", "spend_m",
     "share_of_firm_spend"]].to_csv(
    D + "top10.csv", index=False)
opps[["lever", "count", "unit", "spend_m",
      "per_1pct_m", "share_of_total"]].to_csv(
    D + "opportunities.csv", index=False)
print(os.listdir(D))

['Copy of opportunities.csv', 'kpi.csv', 'category_tier.csv', 'opportunities.csv', 'pareto.csv', 'top10.csv']


In [ ]:
# Chart titles and the headline, built from variables (copy them into Looker Studio)
wk = df[df["category"] == "Works"]
ws = wk["amount_usd"].sum() / tot
t1 = (f"{n80} of {len(s):,} suppliers "
      f"({n80 / len(s):.1%}) hold 80% of firm spend")
t2 = (f"Works carry {ws:.1%} of spend and are "
      f"{gf(wk):.0%} fully competed; Goods only "
      f"{gf(g):.0%} ({gf(ge):.0%} without UN agencies)")
t3 = f"The 10 largest suppliers hold {top10:.1%} of firm spend"
t4 = "Value at stake per 1% saved (illustrative; levers overlap)"
head = (f"{full:.1%} of ${tot / 1e9:.2f}B in Kenya's "
        f"World Bank-financed awards is fully competed, "
        f"but ${dir_ex / 1e6:,.0f}M goes by direct "
        f"selection outside UN agencies. Each 1% saved "
        f"on the five largest awards is worth "
        f"${opps.loc[3, 'per_1pct_m']:.1f}M.")
for t in (t1, t2, t3, t4, head):
    print(t)

180 of 2,931 suppliers (6.1%) hold 80% of firm spend
Works carry 61.9% of spend and are 94% fully competed; Goods only 45% (63% without UN agencies)
The 10 largest suppliers hold 42.6% of firm spend
Value at stake per 1% saved (illustrative; levers overlap)
79.8% of $2.15B in Kenya's World Bank-financed awards is fully competed, but $141M goes by direct selection outside UN agencies. Each 1% saved on the five largest awards is worth $6.1M.


In [ ]:
# Tail check: share of tail suppliers that won only one award
n_aw = firms.groupby("supplier_key").size()
once = (n_aw[tail.index] == 1).mean()
print(round(once, 3), len(tail))

0.812 2751


In [ ]:
# Summary text and four decision statements, all numbers from variables
lm = df["tier"] == "Limited"
lim_n = int(lm.sum())
lim_v = df.loc[lm, "amount_usd"].sum()
loc = lambda d: (d.loc[d["is_kenyan"] == 1,
    "amount_usd"].sum() / d["amount_usd"].sum())
out5 = int((l4["is_kenyan"] == 0).sum())
rfb = int((l4["method"] == "Request for Bids").sum())
problem = (
    f"Where does Kenya's World Bank-financed contract "
    f"spend go, how concentrated is it, and how much is "
    f"competed? {len(df):,} supplier awards worth "
    f"${tot / 1e9:.2f}B, contracts signed "
    f"{df['signing_date'].min()} to "
    f"{df['signing_date'].max()}.")
method = (
    f"I cleaned the data in pandas (supplier names "
    f"merged by a written map, individual consultants "
    f"set aside), queried it in SQL, and measured "
    f"concentration, competition tiers and local share. "
    f"Firm spend excludes individual consultants; "
    f"UN agencies are included.")
findings = (
    f"Spend is concentrated: {n80} of {len(s):,} "
    f"suppliers hold 80% of firm spend and the top 10 "
    f"hold {top10:.1%}. Works are {ws:.1%} of spend "
    f"and {gf(wk):.0%} fully competed; Goods are "
    f"{gf(g):.0%} fully competed ({gf(ge):.0%} without "
    f"UN agencies). Limited-competition awards are "
    f"{lim_n:,} contracts ({lim_n / len(df):.0%}) but "
    f"only {lim_v / tot:.1%} of value.")
local = (
    f"Kenya-registered suppliers receive "
    f"{loc(df):.1%} of value ({loc(ex):.1%} without "
    f"UN agencies), but only {loc(wk):.1%} of Works; "
    f"{out5} of the five largest awards went to "
    f"suppliers registered outside Kenya.")
d1 = (
    f"Decision 1, direct awards: the implementing "
    f"agency's procurement unit should ask for the "
    f"justification of each of the {len(l1)} direct "
    f"awards outside UN agencies "
    f"(${dir_ex / 1e6:,.0f}M, {dir_ex / tot_ex:.1%} of "
    f"non-UN spend) and test whether the next similar "
    f"purchase can be competed. Each 1% saved is worth "
    f"${dir_ex / 1e8:.1f}M. Check first: whether each "
    f"was single-source for a valid reason, such as an "
    f"emergency.")
d2 = (
    f"Decision 2, repeat small awards: the implementing "
    f"agency's procurement unit should ask whether a "
    f"framework agreement fits the {len(l2)} suppliers "
    f"with five or more limited-competition awards "
    f"(${l2['spend'].sum() / 1e6:.1f}M, "
    f"{l2['spend'].sum() / tot:.1%} of total value). "
    f"Each 1% saved is worth "
    f"${l2['spend'].sum() / 1e8:.2f}M, so this is a "
    f"small lever and a cheap one to test. Check first: "
    f"the item, the unit price and the dates of those "
    f"awards.")
d3 = (
    f"Decision 3, the five largest awards: the "
    f"implementing agency's contract managers should put "
    f"the five largest awards, "
    f"{opps.loc[3, 'share_of_total']:.1%} of total value, "
    f"under the closest contract management. Each 1% is "
    f"worth ${opps.loc[3, 'per_1pct_m']:.1f}M. {rfb} of "
    f"the 5 were competed through Request for Bids and "
    f"{5 - rfb} was direct selection, so the lever is "
    f"contract management, not the award. Check first: "
    f"amendments, variations and payment against "
    f"commitment, which this dataset does not hold.")
d4 = (
    f"Decision 4, the long tail: the implementing "
    f"agency's procurement unit should look at lot "
    f"packaging and bidding effort for the {len(tail):,} "
    f"suppliers beyond the 80% line, because {once:.0%} "
    f"of them won only one award "
    f"(${tail.sum() / 1e6:,.0f}M, "
    f"{tail.sum() / tot:.1%} of total value). Each 1% "
    f"saved is worth ${tail.sum() / 1e8:.1f}M. Check "
    f"first: what these one-off purchases are and "
    f"whether similar ones could be grouped into fewer, "
    f"larger lots or a framework agreement. The data has "
    f"no item-level detail, so it cannot show whether "
    f"they are genuinely different needs.")
limit = (
    "Limit: the data has no prices or baseline costs, "
    "so 'value at stake' is what each 1% is worth, not "
    "savings achieved. Amounts are commitments, only "
    "World Bank-financed contracts are included, the "
    "competition tiers are my reading of the method "
    "names, and the levers overlap, so they must not "
    "be added.")

In [ ]:
# Problem, method, findings, local, d1 to d4, limit
print("\n\n".join([problem, method, findings, local,
                   d1, d2, d3, d4, limit]))

Where does Kenya's World Bank-financed contract spend go, how concentrated is it, and how much is competed? 4,839 supplier awards worth $2.15B, contracts signed 2019-07-03 to 2026-10-15.

I cleaned the data in pandas (supplier names merged by a written map, individual consultants set aside), queried it in SQL, and measured concentration, competition tiers and local share. Firm spend excludes individual consultants; UN agencies are included.

Spend is concentrated: 180 of 2,931 suppliers hold 80% of firm spend and the top 10 hold 42.6%. Works are 61.9% of spend and 94% fully competed; Goods are 45% fully competed (63% without UN agencies). Limited-competition awards are 3,374 contracts (70%) but only 6.6% of value.

Kenya-registered suppliers receive 48.5% of value (52.1% without UN agencies), but only 37.2% of Works; 5 of the five largest awards went to suppliers registered outside Kenya.

Decision 1, direct awards: the implementing agency's procurement unit should ask for the justific